# IL Viscosity Prediction Pipeline - Professional Template

**Purpose:** Consolidated, reusable workflow for querying ionic liquid (IL) data and generating viscosity predictions using trained GCNN models.

**Author:** Ziru Huang  
**Date:** October 2025

## Workflow Overview
1. Setup and import utilities
2. Define ionic liquid (cation + anion)
3. Query experimental data from VISPILS database
4. Prepare input files for GCNN prediction
5. Execute model predictions
6. Compare predictions with experimental data
7. Analyze errors and create visualizations
8. Export results

---
## Section 1: Setup and Data Loading

In [ ]:
%load_ext autoreload
%autoreload 2

# Standard libraries
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem

# Custom utilities
import sys
sys.path.append("../scripts/")
from data_utils import dfUtils, merge_with_tolerance
from smiles_utils import sanitized_smiles
from chemprop_utils import create_chemprop_input_files, run_chemprop_prediction, process_chemprop_results

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All imports successful")

---
## Section 2: SMILES Validation and Sanitization

Define the ionic liquid (IL) by specifying cation and anion SMILES strings.

In [ ]:
# ============================================
# USER INPUT: Define your ionic liquid here
# ============================================

# Example: 1-Butyl-3-methylimidazolium tetrafluoroborate
cation_smiles = "CCCCn1cc[n+](C)c1"
anion_smiles = "F[B-](F)(F)F"

# Combine to get IL SMILES
il_smiles = f"{cation_smiles}.{anion_smiles}"
print(f"Original IL SMILES: {il_smiles}")

# Sanitize using RDKit
sanitized_cation_smiles = sanitized_smiles(cation_smiles)
sanitized_anion_smiles = sanitized_smiles(anion_smiles)
sanitized_il_smiles = sanitized_smiles(il_smiles)

print(f"\nSanitized cation: {sanitized_cation_smiles}")
print(f"Sanitized anion:  {sanitized_anion_smiles}")
print(f"Sanitized IL:     {sanitized_il_smiles}")

# Visualize the molecule
mol = Chem.MolFromSmiles(sanitized_il_smiles)
display(mol)

---
## Section 3: Dataset Query and Exploration

In [ ]:
# Load VISPILS experimental dataset
print("Loading VISPILS dataset...")
df = pd.read_csv("../../vispils/data/data_vispils.csv")
df = dfUtils(df)

print("Sanitizing dataset columns...")
df.sanitize_old_df(inplace=True)

print("\n=== Full Dataset Summary ===")
df.data_summary(il_smiles_col="sanitized_il_smiles", temp_col="temperature_k")

In [ ]:
# Query experimental data for the specific IL
df_query = df[df["sanitized_il_smiles"] == sanitized_il_smiles]

if len(df_query) == 0:
    print(f"❌ No data found for IL: {sanitized_il_smiles}")
    print("Please check the SMILES string and try again.")
else:
    df_query = dfUtils(df_query)
    print(f"\n=== Queried IL Data ===")
    df_query.data_summary(il_smiles_col="sanitized_il_smiles", temp_col="temperature_k")

---
## Section 4: Prepare Prediction Input Files

In [ ]:
# Define temperature range for predictions
# Option 1: Use same temperatures as experimental data
# target_temperature_k = sorted(df_query['temperature_k'].unique().tolist())

# Option 2: Define custom temperature range
target_temperature_k = [283, 293, 298.15, 303, 308, 313, 318, 323, 328, 333, 338, 343, 348, 353, 358, 363, 368, 373]

print(f"Prediction temperatures ({len(target_temperature_k)}): {target_temperature_k}\n")

# Create input files
tar_il_smiles = [sanitized_il_smiles] * len(target_temperature_k)

prediction_paths = create_chemprop_input_files(
    il_smiles=tar_il_smiles,
    temperature_k=target_temperature_k,
    input_files_dir="./model_input_files",
    verbose=True
)

---
## Section 5: Execute Model Predictions

### Step 5a: Verify command (dry-run)

In [ ]:
# Run in dry_run mode to verify the command before execution
print("\n" + "="*70)
print("DRY RUN: Verifying prediction command")
print("="*70)

result_dry = run_chemprop_prediction(
    input_files_dir="./model_input_files",
    predict_script_path="../../vispils/predict.py",
    model_checkpoint_path="../../vispils/models/model-corr-5-temp-100-3-2-scale3-constrain-seed42",
    dry_run=True,
    verbose=True
)

print("\n✓ Dry run complete. Review the command above.")
print("  When ready, execute the cell below with dry_run=False")

### Step 5b: Execute actual predictions

In [ ]:
# ⚠️  Uncomment the code below to run actual predictions
# This may take several minutes depending on the model size

# result = run_chemprop_prediction(
#     input_files_dir="./model_input_files",
#     predict_script_path="../../vispils/predict.py",
#     model_checkpoint_path="../../vispils/models/model-corr-5-temp-100-3-2-scale3-constrain-seed42",
#     dry_run=False,
#     verbose=True
# )

---
## Section 6: Results Processing and Comparison

In [ ]:
# Load prediction results
df_results = process_chemprop_results(
    predict_output_path="./model_input_files/predict.csv",
    viscosity_format="log mpas",
)

print(f"\n✓ Loaded {len(df_results)} predictions")
print("\nPrediction Results:")
display(df_results)

In [ ]:
# Convert predictions to log scale
df_results['log_10_pred_viscosity_mpas'] = np.log10(df_results['pred_0'])

# ========================================
# MERGE WITH TEMPERATURE TOLERANCE
# ========================================
# Temperature matching with tolerance handles precision differences
# between experimental and predicted temperatures
TEMPERATURE_TOLERANCE_K = 1.0

print(f"Merging with {TEMPERATURE_TOLERANCE_K} K temperature tolerance...")
print(f"\nExperimental temps ({len(df_query['temperature_k'].unique())}): "
      f"{sorted(df_query['temperature_k'].unique())}")
print(f"Predicted temps ({len(df_results['temperature_k'].unique())}): "
      f"{sorted(df_results['temperature_k'].unique())}")

# Prepare dataframes
df_exp = df_query[['temperature_k', 'viscosity_mpas', 'log_10_viscosity_mpas']].copy()
df_pred = df_results[['temperature_k', 'pred_0', 'log_10_pred_viscosity_mpas']].drop_duplicates().copy()

# Merge with tolerance-based matching
df_comparison = merge_with_tolerance(
    df_left=df_exp,
    df_right=df_pred,
    left_key='temperature_k',
    right_key='temperature_k',
    tolerance=TEMPERATURE_TOLERANCE_K,
    how='left'
)

print(f"\n✓ Merged {len(df_comparison)} data points")
print(f"  Columns: {list(df_comparison.columns)}")

In [ ]:
# ========================================
# CALCULATE PREDICTION ERRORS
# ========================================
df_comparison['error'] = (df_comparison['log_10_pred_viscosity_mpas'] - 
                           df_comparison['log_10_viscosity_mpas'])
df_comparison['abs_error'] = np.abs(df_comparison['error'])
df_comparison['percent_error'] = ((df_comparison['abs_error'] / 
                                    df_comparison['log_10_viscosity_mpas']) * 100)

# Print comprehensive error statistics
print("\n" + "="*70)
print("PREDICTION ERROR ANALYSIS")
print("="*70)
print(f"Mean Absolute Error (log units):    {df_comparison['abs_error'].mean():.4f}")
print(f"RMSE (log units):                   {np.sqrt((df_comparison['error']**2).mean()):.4f}")
print(f"Max Error (log units):              {df_comparison['abs_error'].max():.4f}")
print(f"Min Error (log units):              {df_comparison['abs_error'].min():.4f}")
print(f"Mean Percent Error:                 {df_comparison['percent_error'].mean():.2f}%")
print(f"Number of matched data points:      {len(df_comparison)}")
print("="*70)

---
## Section 7: Error Analysis and Visualization

In [ ]:
# Create output directories if they don't exist
Path("./figures").mkdir(exist_ok=True)
Path("../results").mkdir(exist_ok=True)

# Define visualization styles
subset_styles = {
    'Experimental': {'marker': 'o', 'color': 'C0', 'label': 'Experimental',
                     's': 100, 'alpha': 0.7, 'edgecolors': 'blue', 'linewidth': 2},
    'Predicted': {'marker': '^', 'color': 'C1', 'label': 'Predicted',
                  's': 100, 'alpha': 0.7, 'edgecolors': 'red', 'linewidth': 2}
}

temperature_col = 'temperature_k'

In [ ]:
# ============================================
# Figure 1: Comprehensive 4-subplot analysis
# ============================================
fig, axes = plt.subplots(2, 2, figsize=(12, 10), dpi=150)
fig.suptitle(f"GCNN Viscosity Prediction Analysis\nIL: {sanitized_il_smiles}", 
             fontsize=14, fontweight='bold')

# Plot 1: Temperature vs Viscosity (linear scale)
ax = axes[0, 0]
ax.scatter(df_comparison[temperature_col], df_comparison['viscosity_mpas'], 
          **subset_styles['Experimental'])
ax.scatter(df_comparison[temperature_col], df_comparison['pred_0'], 
          **subset_styles['Predicted'])
ax.set_xlabel('Temperature (K)', fontsize=11)
ax.set_ylabel('Viscosity (mPa·s)', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title('Viscosity vs Temperature')

# Plot 2: Temperature vs log Viscosity
ax = axes[0, 1]
ax.scatter(df_comparison[temperature_col], df_comparison['viscosity_mpas'], 
          **subset_styles['Experimental'])
ax.scatter(df_comparison[temperature_col], df_comparison['pred_0'], 
          **subset_styles['Predicted'])
ax.set_yscale('log')
ax.set_xlabel('Temperature (K)', fontsize=11)
ax.set_ylabel('log₁₀(Viscosity) [mPa·s]', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title('log Viscosity vs Temperature')

# Plot 3: Parity Plot
ax = axes[1, 0]
min_val = min(df_comparison['log_10_viscosity_mpas'].min(), 
              df_comparison['log_10_pred_viscosity_mpas'].min())
max_val = max(df_comparison['log_10_viscosity_mpas'].max(), 
              df_comparison['log_10_pred_viscosity_mpas'].max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='Perfect Prediction')
ax.scatter(df_comparison['log_10_viscosity_mpas'], 
          df_comparison['log_10_pred_viscosity_mpas'],
          **subset_styles['Predicted'])
ax.set_xlabel('Experimental log₁₀(Viscosity) [mPa·s]', fontsize=11)
ax.set_ylabel('Predicted log₁₀(Viscosity) [mPa·s]', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)
mae = df_comparison['abs_error'].mean()
ax.set_title(f"Parity Plot (MAE: {mae:.4f})")

# Plot 4: Error Distribution by Temperature
ax = axes[1, 1]
colors = ['green' if x > -0.5 else 'red' for x in df_comparison['error']]
ax.bar(range(len(df_comparison)), df_comparison['error'], color=colors, alpha=0.7)
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.axhline(y=df_comparison['error'].mean(), color='blue', linestyle='--', 
           linewidth=2, label='Mean Error')
ax.set_xlabel('Temperature Index', fontsize=11)
ax.set_ylabel('Error (log units)', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_title('Prediction Error by Temperature')

plt.tight_layout()
fig.savefig("./figures/viscosity_analysis.png", dpi=300, bbox_inches='tight')
print("✓ Figure saved: ./figures/viscosity_analysis.png")
plt.show()

In [ ]:
# ============================================
# Figure 2: Simplified 2-subplot view
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=150)
fig.suptitle(f"GCNN Viscosity Prediction\nIL: {sanitized_il_smiles}", 
             fontsize=14, fontweight='bold')

# Plot 1: Log Viscosity vs Temperature
ax = axes[0]
ax.scatter(df_comparison[temperature_col], df_comparison['viscosity_mpas'], 
          **subset_styles['Experimental'])
ax.scatter(df_comparison[temperature_col], df_comparison['pred_0'], 
          **subset_styles['Predicted'])
ax.set_yscale('log')
ax.set_xlabel('Temperature (K)', fontsize=11)
ax.set_ylabel('log₁₀(Viscosity) [mPa·s]', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title('log Viscosity vs Temperature')

# Plot 2: Parity Plot
ax = axes[1]
min_val = min(df_comparison['log_10_viscosity_mpas'].min(), 
              df_comparison['log_10_pred_viscosity_mpas'].min())
max_val = max(df_comparison['log_10_viscosity_mpas'].max(), 
              df_comparison['log_10_pred_viscosity_mpas'].max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='Perfect Prediction')
ax.scatter(df_comparison['log_10_viscosity_mpas'], 
          df_comparison['log_10_pred_viscosity_mpas'],
          **subset_styles['Predicted'])
ax.set_xlabel('Experimental log₁₀(Viscosity) [mPa·s]', fontsize=11)
ax.set_ylabel('Predicted log₁₀(Viscosity) [mPa·s]', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)
mae = df_comparison['abs_error'].mean()
ax.set_title(f"Parity Plot (MAE: {mae:.4f})")

plt.tight_layout()
fig.savefig("./figures/viscosity_parity.png", dpi=300, bbox_inches='tight')
print("✓ Figure saved: ./figures/viscosity_parity.png")
plt.show()

---
## Section 8: Export Results

In [ ]:
# Save detailed comparison results
output_csv = "../results/viscosity_comparison.csv"
df_comparison.to_csv(output_csv, index=False)
print(f"✓ Results saved to: {output_csv}")
print(f"\nDataFrame shape: {df_comparison.shape}")
print(f"Columns: {list(df_comparison.columns)}")

In [ ]:
# Create a summary report
summary = f"""
{'='*70}
IONIC LIQUID VISCOSITY PREDICTION - SUMMARY REPORT
{'='*70}

IL INFORMATION:
  Original SMILES:    {il_smiles}
  Sanitized SMILES:   {sanitized_il_smiles}
  Cation:             {sanitized_cation_smiles}
  Anion:              {sanitized_anion_smiles}

EXPERIMENTAL DATA:
  Total data points:  {len(df_query)}
  Temp range:         {df_query['temperature_k'].min():.1f} - {df_query['temperature_k'].max():.1f} K
  Viscosity range:    {df_query['viscosity_mpas'].min():.2f} - {df_query['viscosity_mpas'].max():.2f} mPa·s

PREDICTION STATISTICS:
  Matched data points: {len(df_comparison)}
  Temperature tolerance: {TEMPERATURE_TOLERANCE_K} K
  MAE (log units):     {df_comparison['abs_error'].mean():.4f}
  RMSE (log units):    {np.sqrt((df_comparison['error']**2).mean()):.4f}
  Max error:           {df_comparison['abs_error'].max():.4f}
  Mean % error:        {df_comparison['percent_error'].mean():.2f}%

OUTPUT FILES:
  Results CSV:         ../results/viscosity_comparison.csv
  Analysis figure:     ./figures/viscosity_analysis.png
  Parity figure:       ./figures/viscosity_parity.png

{'='*70}
"""

print(summary)

# Save summary to file
with open("../results/prediction_summary.txt", "w") as f:
    f.write(summary)

print("✓ Summary saved to: ../results/prediction_summary.txt")